# FX Pairs — Training Pipeline V7 (Kaggle T4, resumable)

Every change from V6 is marked in the code as `── Cn ──` with three lines:
**V6** what it was, **WHY** it was a problem, **NEW** what it does now.
Nothing is changed silently — if you want to know why a line looks different,
the reason is directly above it.

---

### The resume system — the point of this rewrite

Kaggle kills a GPU session after 9 hours, and a closed tab ends one sooner.
V6 lost everything when that happened; your TFT died at epoch 8 of 50 and all
of it was thrown away. Here the run is a list of **units** — one `(fold, model)`
pair each. After every unit, the whole run folder is zipped and pushed to a
private Kaggle Dataset *and* to Google Drive, alongside a `run_state.json`
ledger of what is finished.

```
session 1  ▶ f0_NBEATSx ✅  f0_NHITS ✅  f0_PatchTST ✅  f0_TFT ✅
           ▶ f1_NBEATSx ✅   ← session killed here
session 2  ✓ f0_NBEATSx skipped   ✓ f0_NHITS skipped   ✓ f0_PatchTST skipped
           ✓ f0_TFT skipped      ✓ f1_NBEATSx skipped
           ▶ f1_NHITS ✅   ▶ f1_PatchTST ✅   …continues
```

Rerun the notebook with the same `RUN_NAME` and it picks up at the exact unit
that was interrupted. A 12-hour run simply takes two sessions.

---

### Change index

| # | Cell | Change | Why it mattered |
|---|---|---|---|
| C1 | 1 | Spacetimeformer removed | Never installed → silently skipped; when it did run it trained on one sample for 20 gradient steps |
| C2 | 1 | GPU printed before anything | V6 ran at 0.52 it/s = CPU speed = 20 h for the run |
| C3 | 2 | Reads the slim parquet | V6's Cell B wrote `leg1_macro_growth`, the new data has `macro_leg1_growth` → 30 NaN columns → crash |
| C4 | 2 | `RUN_NAME` as resume key | V6 had no concept of a run |
| C5 | 2 | Kaggle Dataset + Drive mirrors | Kaggle can't mount Drive; `/kaggle/working` is wiped when a session ends |
| C6 | 2 | Session + per-model wall clocks | V6 could only ever end by being killed |
| C7 | 2 | horizon 5 → 1 | Target autocorrelation is 0.041; 4 of 5 predictions were into pure noise |
| C8 | 2 | Rolling-origin folds | V6's test set was 100 rows, ±21% noise |
| C9 | 2 | PatchTST kept as control | It genuinely cannot take exogenous inputs |
| C10 | 2 | batch 128 + fp16 + real step budget | 300 steps is a fraction of one pass over 55,540 rows |
| **C11** | **3** | **Whole persistence layer (new)** | **V6's Cell 16 deleted previous outputs on rerun — the opposite of resuming** |
| C12 | 4 | Schema contract asserted at load | V6 discovered missing columns as silent NaN, crashing six cells later |
| C13 | 4 | Target grouped on both sides | V6's ungrouped `.shift(-1)` was correct only by luck |
| C14 | 4 | Calendar features generated | V6 found 0 known-future columns |
| C15 | 4 | Fold layout checked upfront | Finding out 40 min into a fit is expensive |
| C16 | 5 | `*_lead1` dropped as duplicates | V6 dropped them as a "leak" — wrong reason, right action |
| C17 | 5 | Known-future list populated | TFT's known/unknown pathway was switched off |
| C18 | 5 | Regimes categorical / one-hot | `factorize()` floats told the model state 2 is twice state 1 |
| **C19** | **5** | **`zscore` + `spread` added as features** | **Strongest honest signal in the file (−0.095) and it was in no list** |
| C20 | 5 | Constant/NaN columns dropped | V6 fed `event_flag` + 2 lags, all three constant |
| C21 | 6 | 3 × 250-day walked-forward test | One market week decided V6's ranking |
| C22 | 7 | Three baselines | Predict-zero scores 0.110; V6 had nothing to compare against |
| C23 | 7 | Bootstrap resampling groups | Days inside a group are correlated; row bootstrap would lie |
| C24 | 7 | Directional accuracy | Sign matters more than magnitude for a trade |
| C25 | 8 | Accelerator + precision explicit | V6 left it to auto-detect and got CPU |
| C26 | 8 | `max_time` on every fit | A slow model costs one unit, not the session |
| C27–C29 | 9 | TFT feature wiring | Empty known_reals, missing z-score, float regimes |
| C30 | 9 | Validation = ~120 windows/group | V6 validated, checkpointed and early-stopped on **20 samples** |
| C31 | 9 | Rolling test window | V6's `predict=True` gave one window per group |
| C32 | 10 | Flat trainer kwargs | `trainer_kwargs={...}` → `TypeError` |
| C33 | 10 | `valid_loss=MQLoss` | `MQLoss` + `valid_loss=MAE()` → raises on **NBEATSx**, the first model built |
| C34 | 10 | Real step budget | Patience counted validation checks; early stopping could never fire |
| C35 | 10 | Robust scaler | Heavy-tailed features; crisis days were setting the scale |
| C36 | 10 | PatchTST fixed | `EXOGENOUS_HIST=False` raises, and `d_model`/`e_layers` aren't parameters |
| C37 | 10 | Integer `ds`, `freq=1` | `freq="B"` assumes no market holidays |
| C38 | 10 | `cross_validation` for the test | `predict()` gave 5 rows per group |
| C39–C41 | 11 | Unit ledger, isolation, baselines first | One PatchTST exception killed NBEATSx and NHITS too |
| C42–C43 | 12 | Nothing vanishes; conclusions printed | `dropna()` hid a model that never ran |
| C44 | 13 | Manifest records feature lists | "Why did that run score differently" answerable later |

---

**Set expectations before you run it.** The honest test setup will produce
*worse* numbers than V6 would have. That is the leakage and the 100-row
lottery leaving. What you get back is a leaderboard you can defend.


## Step 0 — One-time setup

**Accelerator:** right panel → Settings → Accelerator → **GPU T4 x2**, and Internet **On**.

**Data:** upload the slim parquet from the corrected data pipeline as a Kaggle Dataset,
then `+ Add Data` it here. Adjust `DATA_CANDIDATES` in the config cell to match its path.

**Two Kaggle Secrets** (Add-ons → Secrets → Add secret):

| Secret name | What to paste | Needed for |
|---|---|---|
| `KAGGLE_JSON` | the whole contents of your `kaggle.json` (Kaggle → Account → Create New API Token) | the resume path |
| `GDRIVE_SA_JSON` | a Google Cloud **service-account** key JSON | the Drive copy |

For the Drive one: create a service account in Google Cloud, enable the Drive API,
download its JSON key, then **share your Drive folder with that service account's email
address** (Editor). The folder ID is the last part of the folder's URL. If you skip this,
set `USE_DRIVE = False` — the Kaggle Dataset path alone is enough to resume.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1 — INSTALL                                            [CHANGES C1, C2]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C1 ── spacetimeformer removed from the stack ──────────────────────────
#   V6:  `import spacetimeformer as stf` in Cell J, but the install line only
#        installed lightning / neuralforecast / pytorch_forecasting.
#   WHY: the import always failed, so STF_AVAILABLE was False, the whole
#        Spacetimeformer block silently skipped, stf_test_mae stayed None, and
#        the leaderboard's .dropna() removed the row. A model was "in the
#        comparison" that had never run once, and nothing said so.
#        Even when it did run, its training loop fed the entire history as ONE
#        sample (shape (1, 2717, 20)), did 20 gradient steps total, and
#        truncated the label to the model's output length -- comparing a
#        5-step forecast against the first five days of 2013.
#   NEW: dropped entirely. Its slot in the leaderboard is taken by a
#        mean-reversion baseline, which answers a more useful question:
#        "does any neural net beat a two-parameter straight line?"
#
# ── C2 ── the GPU is now verified, not assumed ────────────────────────────
#   V6:  never checked. pl.Trainer() was constructed with no accelerator
#        argument and left to auto-detect.
#   WHY: your saved output shows 847 batches/epoch at 0.52 it/s -- about 27
#        minutes per epoch, 20+ hours for 50 epochs. That is CPU speed for a
#        TFT this small. The run died at epoch 8 because the session ended.
#   NEW: print the device before anything else. If this says NO GPU, stop and
#        fix it; nothing below will finish otherwise.
# ═══════════════════════════════════════════════════════════════════════════
!pip install -q neuralforecast pytorch-forecasting
!pip install -q kaggle google-api-python-client google-auth

import torch, sys
print("python ", sys.version.split()[0])
print("torch  ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


## Step 1 — Config

The only cell you normally edit. `RUN_NAME` is the resume key: keep it the same to
continue a run, change it to start over from zero.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2 — CONFIG                                   [CHANGES C3 … C10]
# ═══════════════════════════════════════════════════════════════════════════
import os, json, time, shutil, warnings
warnings.filterwarnings("ignore")

# ── C3 ── input is the slim parquet, not the 616-column CSV ───────────────
#   V6:  CSV_PATH = "/content/drive/MyDrive/dcc_garch_calander_gdelt_final.csv"
#        and Cell B rebuilt leg1/leg2/leg3 columns from the wide ticker names.
#   WHY: the data pipeline now does that remapping upstream and emits
#        macro_leg1_growth. Cell B looked for macro_AUDCAD=X_growth and wrote
#        leg1_macro_growth -- different names on both sides. Run the new data
#        through the old notebook and all 30 macro columns become NaN, then
#        TimeSeriesDataSet raises on NaN and TFT dies at construction.
#        Two notebooks independently owned the same mapping. Now only one does.
#   NEW: read the slim file directly, and assert the schema (see cell 4).
DATA_CANDIDATES = [
    "/kaggle/input/fx-features/dcc_garch_calander_gdelt_final.parquet",
    "/kaggle/input/fx-features/dcc_garch_calander_gdelt_final.csv",
]

# ── C4 ── RUN_NAME is the resume key ──────────────────────────────────────
#   V6:  no concept of a run. A dead Colab session meant starting over.
#   NEW: everything resumable lives under RUN_DIR and is keyed by this name.
#        Same name  -> continue where the last session stopped.
#        New name   -> start from zero, old snapshot untouched.
RUN_NAME = "fx_v7_run1"

WORK_DIR = "/kaggle/working"
RUN_DIR  = f"{WORK_DIR}/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)

# ── C5 ── two checkpoint mirrors instead of a mounted Drive ───────────────
#   V6:  drive.mount() and direct writes to /content/drive/MyDrive/fx_models/.
#   WHY: Kaggle cannot mount Drive the way Colab does -- there is no browser
#        to click through the Google login, and a scheduled/resumed run must
#        work with nobody watching. /kaggle/working is also wiped when an
#        interactive session ends unless you remember to Save Version.
#   NEW: (A) a private Kaggle Dataset, written with the API key -- this is the
#        path that actually restores the run. (B) Google Drive through a
#        service account, so results still land next to your data pipeline.
#        Either one alone is enough; both are on by default.
USE_KAGGLE_CKPT   = True
KAGGLE_USERNAME   = "your-kaggle-username"    # <-- EDIT
CKPT_DATASET_SLUG = "fx-training-checkpoints" # created automatically first run

USE_DRIVE         = True
DRIVE_FOLDER_ID   = "PASTE_YOUR_DRIVE_FOLDER_ID"   # <-- EDIT

# ── C6 ── session and per-model wall clocks ───────────────────────────────
#   V6:  max_epochs=50 with no time limit at all, on a platform that kills
#        the session at a fixed hour. The run could only ever end by being
#        killed, which is exactly what happened at epoch 8.
#   NEW: the loop refuses to start a unit it cannot finish, and each single
#        model gets a hard cap. The session ends by choice, after a
#        checkpoint, instead of being cut off mid-epoch.
SESSION_BUDGET_HOURS = 7.5          # Kaggle kills GPU sessions at 9h
SESSION_START        = time.time()
SESSION_DEADLINE     = SESSION_START + SESSION_BUDGET_HOURS * 3600
PER_UNIT_BUDGET_MIN  = 55

# ── C7 ── horizon 5 -> 1 ──────────────────────────────────────────────────
#   V6:  horizon = 5.
#   WHY: the target's own lag-1 autocorrelation is 0.041 -- it is close to
#        white noise one day out, and there is no reason steps 2-5 carry more.
#        Four of every five predictions were being made into a region where
#        nothing is predictable, and those four dominated the MAE. If the
#        decision is "do I put the trade on tomorrow", h=1 IS the decision --
#        and it multiplies the number of scorable test points by five for free.
HORIZON    = 1
INPUT_SIZE = 60      # unchanged from V6: 60 days of history

# ── C8 ── one 5-day test window -> rolling-origin folds ───────────────────
#   V6:  test = last 5 rows per group = 100 rows total, one week of Jan 2024.
#   WHY: measured on your data, the standard error of an MAE computed on 100
#        rows is +/-9.6%, and +/-21% once you allow for the five days inside a
#        group being correlated. Any leaderboard gap under ~20% was a coin
#        flip. You cannot rank five architectures on that, however well each
#        one is trained.
#   NEW: 3 windows of 250 trading days each, walked forward. Each fold trains
#        only on data before its own validation window, so nothing from a
#        fold's future reaches its training set.
N_FOLDS    = 3
TEST_SIZE  = 250
VAL_SIZE   = 120     # V6 used 60; doubled so validation is a real sample

# ── C9 ── model list ──────────────────────────────────────────────────────
#   PatchTST stays, but as the univariate control (see C24) rather than a
#   like-for-like entry. Baselines always run and are not optional.
MODELS = ["NBEATSx", "NHITS", "PatchTST", "TFT"]

# ── C10 ── throughput settings ────────────────────────────────────────────
#   V6:  batch_size=64, no precision setting, max_steps=300.
#   WHY: 300 steps at NeuralForecast's default batch covers a small fraction
#        of one pass over 55,540 rows -- those models were barely trained.
#        And early_stop_patience_steps=5 counts VALIDATION CHECKS, not steps;
#        with val_check_steps=50 that meant patience only expired after 250
#        steps, so early stopping could essentially never fire inside a
#        300-step budget. The two settings were not doing what their names
#        suggest.
#   NEW: bigger batch + fp16 for the T4, and a step budget the models can
#        actually learn inside, bounded by the wall clock rather than by an
#        arbitrary step count.
SEED         = 42
BATCH_SIZE   = 128
MAX_EPOCHS   = 30        # TFT
MAX_STEPS    = 4000      # NeuralForecast
PRECISION    = "16-mixed"

print(f"run={RUN_NAME}  h={HORIZON}  folds={N_FOLDS}  "
      f"test={TEST_SIZE}d  val={VAL_SIZE}d  budget={SESSION_BUDGET_HOURS}h")


## Step 2 — Checkpoint layer

Restores the previous snapshot if there is one, and defines `sync_out()` — the function
called after every unit.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3 — PERSISTENCE                                       [CHANGE C11 — new cell]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C11 ── nothing like this existed in V6 ────────────────────────────────
#   V6:  Cell 0 created folders on a mounted Drive and each model wrote its
#        checkpoint there when it finished. There was no record of WHICH
#        models had finished, so a rerun re-executed everything from the top.
#        Cell 16 (the "full-folder reset") actively deleted all model outputs
#        at the start of a rerun -- the opposite of resuming.
#   WHY: your TFT died at epoch 8 of 50 and every hour of it was thrown away.
#        On Kaggle the session limit is hard, so a 12-hour run must be able to
#        span sessions or it can never complete at all.
#   NEW: the run is a list of UNITS -- one (fold, model) pair each. Everything
#        goes into RUN_DIR. A small run_state.json in there is the ledger of
#        which units are DONE and which FAILED. After every unit, RUN_DIR is
#        zipped and pushed to a private Kaggle Dataset and to Drive. On
#        startup the newest zip is pulled back and unpacked, so the ledger
#        already knows what is finished and the main loop skips it.
#
#        session 1  > f0_NBEATSx OK  f0_NHITS OK  f0_PatchTST OK ... killed
#        session 2  ~ f0_NBEATSx skip  ~ f0_NHITS skip  > f0_TFT ... continues
#
#        Nothing that finished is ever computed twice.
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, glob, io, os, json, shutil, time

SNAPSHOT = f"{WORK_DIR}/{RUN_NAME}_snapshot.zip"


# ── Secrets ────────────────────────────────────────────────────────────────
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as e:
        print(f"   secret '{name}' unavailable: {type(e).__name__}")
        return None


# ── C11b ── accept BOTH of Kaggle's token formats, and prove it works ─────
#   WHY: Kaggle's Settings -> API page now shows two buttons next to each
#        other -- "Generate New Token" (a bare token string) and, under
#        "Legacy API Credentials", "Create Legacy API Key" (the kaggle.json
#        file with username + key). The CLI wants them stored in DIFFERENT
#        places. Pasting the wrong one into the secret used to write garbage
#        to kaggle.json, and the only symptom was `kaggle=no` printed once,
#        40 minutes later, next to a checkpoint that never uploaded.
#   NEW: detect which format the secret holds and store it correctly, then
#        immediately make one real API call to prove the key is accepted.
#        You learn in 5 seconds, not after half a session of unsaved work.
def _setup_kaggle_cli():
    raw = _secret("KAGGLE_JSON")
    if not raw:
        return False
    raw = raw.strip()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    if raw.startswith("{"):                       # legacy kaggle.json
        p = os.path.expanduser("~/.kaggle/kaggle.json")
    else:                                          # new bare token string
        p = os.path.expanduser("~/.kaggle/access_token")
        os.environ["KAGGLE_API_TOKEN"] = raw
    with open(p, "w") as f:
        f.write(raw)
    os.chmod(p, 0o600)
    print(f"   kaggle credential stored as {os.path.basename(p)}")
    return True


KAGGLE_READY = _setup_kaggle_cli() if USE_KAGGLE_CKPT else False

if KAGGLE_READY:
    # `kaggle datasets list -m` lists YOUR OWN datasets, so it only returns 0
    # when the credential is actually accepted. No extra flags: --page-size
    # exists in some CLI versions and not others, and an unrecognised flag
    # makes a perfectly good key look rejected.
    _t = subprocess.run(["kaggle", "datasets", "list", "-m"],
                        capture_output=True, text=True)
    _err = ((_t.stderr or "") + (_t.stdout or "")).strip()
    if _t.returncode == 0:
        print("   ✅ Kaggle API accepted — checkpoints WILL be backed up")
    elif "unrecognized arguments" in _err or _err.startswith("usage:"):
        # the CLI on this image takes different flags — that says nothing
        # about the key, so don't disable checkpointing over it
        print("   ⚠️  could not verify the key (CLI flag mismatch), continuing "
              "anyway — watch the first 'synced ... kaggle=ok/no' line below")
    else:
        KAGGLE_READY = False
        print("   ❌ Kaggle API key rejected — CHECKPOINTS WILL NOT BE SAVED.\n"
              "      Fix: kaggle.com/settings/api -> 'Legacy API Credentials'\n"
              "           -> 'Create Legacy API Key' -> open the downloaded\n"
              "           kaggle.json and paste its WHOLE contents into the\n"
              "           KAGGLE_JSON secret (Add-ons -> Secrets).\n"
              f"      detail: {_err[-300:]}")
elif USE_KAGGLE_CKPT:
    print("   ❌ No KAGGLE_JSON secret found — CHECKPOINTS WILL NOT BE SAVED.\n"
          "      A killed session will lose everything since the last unit.")


# ── Google Drive (service account) ─────────────────────────────────────────
class Drive:
    """Minimal Drive client: upload-or-replace one file, download one file."""

    def __init__(self, folder_id):
        self.svc, self.folder = None, folder_id
        raw = _secret("GDRIVE_SA_JSON")
        if not raw:
            return
        try:
            from google.oauth2.service_account import Credentials
            from googleapiclient.discovery import build
            creds = Credentials.from_service_account_info(
                json.loads(raw), scopes=["https://www.googleapis.com/auth/drive"])
            self.svc = build("drive", "v3", credentials=creds, cache_discovery=False)
        except Exception as e:
            print(f"   Drive disabled: {type(e).__name__}: {e}")

    @property
    def ok(self):
        return self.svc is not None and self.folder and "PASTE" not in str(self.folder)

    def _find(self, name):
        q = f"'{self.folder}' in parents and name='{name}' and trashed=false"
        r = self.svc.files().list(q=q, fields="files(id)", pageSize=1).execute()
        f = r.get("files", [])
        return f[0]["id"] if f else None

    def put(self, path, name=None):
        if not self.ok:
            return False
        from googleapiclient.http import MediaFileUpload
        name = name or os.path.basename(path)
        media = MediaFileUpload(path, resumable=True)
        try:
            fid = self._find(name)
            if fid:
                self.svc.files().update(fileId=fid, media_body=media).execute()
            else:
                self.svc.files().create(
                    body={"name": name, "parents": [self.folder]},
                    media_body=media, fields="id").execute()
            return True
        except Exception as e:
            print(f"   Drive upload failed: {type(e).__name__}: {e}")
            return False

    def get(self, name, dest):
        if not self.ok:
            return False
        from googleapiclient.http import MediaIoBaseDownload
        try:
            fid = self._find(name)
            if not fid:
                return False
            req = self.svc.files().get_media(fileId=fid)
            with io.FileIO(dest, "wb") as fh:
                dl = MediaIoBaseDownload(fh, req)
                done = False
                while not done:
                    _, done = dl.next_chunk()
            return True
        except Exception as e:
            print(f"   Drive download failed: {type(e).__name__}: {e}")
            return False


DRIVE = Drive(DRIVE_FOLDER_ID) if USE_DRIVE else None


# ── Kaggle Dataset mirror ──────────────────────────────────────────────────
def _kaggle_push(msg):
    if not KAGGLE_READY:
        return False
    stage = f"{WORK_DIR}/_ckpt_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage, exist_ok=True)
    shutil.copy(SNAPSHOT, f"{stage}/{RUN_NAME}_snapshot.zip")
    json.dump({"title": CKPT_DATASET_SLUG,
               "id": f"{KAGGLE_USERNAME}/{CKPT_DATASET_SLUG}",
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{stage}/dataset-metadata.json", "w"))

    ver = subprocess.run(["kaggle", "datasets", "version", "-p", stage,
                          "-m", msg, "--dir-mode", "zip"],
                         capture_output=True, text=True)
    if ver.returncode == 0:
        return True
    # first ever push: the dataset does not exist yet
    new = subprocess.run(["kaggle", "datasets", "create", "-p", stage,
                          "--dir-mode", "zip"], capture_output=True, text=True)
    if new.returncode == 0:
        return True
    print("   Kaggle push failed:", (ver.stderr or "")[-200:], (new.stderr or "")[-200:])
    return False


def _kaggle_pull(dest_zip):
    if not KAGGLE_READY:
        return False
    tmp = f"{WORK_DIR}/_ckpt_pull"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp, exist_ok=True)
    r = subprocess.run(["kaggle", "datasets", "download", "-d",
                        f"{KAGGLE_USERNAME}/{CKPT_DATASET_SLUG}",
                        "-p", tmp, "--unzip"], capture_output=True, text=True)
    if r.returncode != 0:
        return False
    hits = glob.glob(f"{tmp}/**/{RUN_NAME}_snapshot.zip", recursive=True)
    if not hits:
        return False
    shutil.copy(hits[0], dest_zip)
    return True


# ── The two functions the rest of the notebook calls ───────────────────────
# sync_out() is called after EVERY unit. It is the single line that decides
# whether a killed session costs you 40 minutes or 8 hours.
def sync_out(msg="checkpoint"):
    """Zip RUN_DIR and mirror it to Kaggle + Drive. Safe to call often."""
    shutil.make_archive(SNAPSHOT[:-4], "zip", RUN_DIR)
    size = os.path.getsize(SNAPSHOT) / 1e6
    k = _kaggle_push(msg) if USE_KAGGLE_CKPT else False
    d = DRIVE.put(SNAPSHOT) if (USE_DRIVE and DRIVE) else False
    print(f"   ⤴ synced {size:.1f} MB  |  kaggle={'ok' if k else 'no'}  drive={'ok' if d else 'no'}")


def restore_in():
    """Pull the newest snapshot back into RUN_DIR. Called once at startup."""
    tmp = f"{WORK_DIR}/_restore.zip"
    got = False
    if USE_KAGGLE_CKPT and _kaggle_pull(tmp):
        got = True
        src = "kaggle dataset"
    elif USE_DRIVE and DRIVE and DRIVE.get(f"{RUN_NAME}_snapshot.zip", tmp):
        got = True
        src = "google drive"
    # A snapshot attached manually via "+ Add Data" also works:
    if not got:
        for p in glob.glob(f"/kaggle/input/**/{RUN_NAME}_snapshot.zip", recursive=True):
            shutil.copy(p, tmp); got = True; src = "attached input"; break
    if not got:
        print("   no previous snapshot found — starting a fresh run")
        return False
    shutil.unpack_archive(tmp, RUN_DIR)
    print(f"   ⤵ restored previous run from {src}")
    return True


# ── Run state: the ledger of what is already finished ──────────────────────
# `done`   -> unit key : timestamp. Skipped on the next run.
# `failed` -> unit key : the exception text. Also skipped, so one broken model
#             cannot burn a whole session; delete the entry to retry it.
STATE_PATH = f"{RUN_DIR}/run_state.json"


def load_state():
    if os.path.exists(STATE_PATH):
        return json.load(open(STATE_PATH))
    return {"run": RUN_NAME, "done": {}, "failed": {}, "created": time.time()}


def save_state(state):
    json.dump(state, open(STATE_PATH, "w"), indent=2)


restore_in()
STATE = load_state()
print(f"   units already complete: {len(STATE['done'])}"
      + (f" | previously failed: {len(STATE['failed'])}" if STATE["failed"] else ""))


## Step 3 — Data, features, folds, metrics

Load and validate the dataset, build the target, sort every column into known-future /
historical / categorical, lay out the rolling-origin folds, and define the scoring.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4 — LOAD, VALIDATE, BUILD TARGET                   [CHANGES C12 … C15]
# ═══════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, random, torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# (unchanged from V6 — the one-seed-set-once decision was already correct)

DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
assert DATA_PATH, f"No dataset found. Looked in:\n  " + "\n  ".join(DATA_CANDIDATES)

df = (pd.read_parquet(DATA_PATH) if DATA_PATH.endswith(".parquet")
      else pd.read_csv(DATA_PATH))
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["group_id", "Date"]).reset_index(drop=True)
print(f"loaded {DATA_PATH}\n   {df.shape[0]:,} rows x {df.shape[1]} cols, "
      f"{df['group_id'].nunique()} groups, {df['Date'].min().date()} -> {df['Date'].max().date()}")

# ── C12 ── a real schema contract ─────────────────────────────────────────
#   V6:  one assert -- `assert "zscore" in df_raw.columns`. Everything else
#        was discovered by pattern matching in Cell B, and anything not found
#        was created as an all-NaN column with a printed warning.
#   WHY: a printed warning in a 40-cell notebook is invisible. The NaN then
#        travelled six cells before surfacing as a crash inside
#        TimeSeriesDataSet, with an error that says nothing about which
#        column was missing or why.
#   NEW: the names this notebook actually depends on are listed once, checked
#        at load time, and the failure message prints the missing ones and
#        tells you what to do. Fail early, fail loud, fail with the answer.
REQUIRED = ["Date", "group_id", "zscore", "spread",
            "rho_leg1_leg2", "rho_leg1_leg3", "rho_leg2_leg3",
            "sigma_leg1", "sigma_leg2", "sigma_leg3",
            "macro_leg1", "macro_leg2", "macro_leg3",
            "shock_events_diff", "avg_tone_diff",
            "regime", "regime_vol", "regime_macro"]
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, (
    "Dataset is missing columns this notebook depends on:\n  "
    + "\n  ".join(missing)
    + "\n\nThis notebook expects the SLIM leg1/leg2/leg3 schema from the "
      "corrected data pipeline. If you are still on the old 616-column CSV, "
      "regenerate the dataset first.")
print("   ✅ schema contract satisfied")

# ── C13 ── the target is now grouped on BOTH sides ────────────────────────
#   V6:  df.groupby("group_id")["zscore"].diff().shift(-1)
#                                                ^^^^^^^^
#        The .diff() is grouped. The .shift(-1) is NOT -- it shifts the whole
#        column across group boundaries.
#   WHY: I traced it and it happens to give the right answer, because the
#        grouped .diff() leaves a NaN at each group's first row and the
#        cross-boundary value lands exactly on that NaN, which then gets
#        dropped. Correct by luck, not by construction. Change the sort
#        order, or swap diff() for something else, and it silently starts
#        assigning one group's move as another group's label.
#   NEW: grouped on both sides. Same numbers today, correct for the right
#        reason, and it stays correct if anything upstream changes.
df["target"] = (df.groupby("group_id")["zscore"].diff()
                  .groupby(df["group_id"]).shift(-1))
df = df.dropna(subset=["target"]).reset_index(drop=True)

# ── C14 ── calendar features, generated here ──────────────────────────────
#   V6:  Cell B scanned for column names containing "dow", "month", "holiday"
#        and printed:  calendar_cols (0) -- treated as KNOWN-FUTURE: []
#   WHY: the data pipeline never produced any, so the detector correctly found
#        nothing. But TFT's known/unknown split and NeuralForecast's
#        futr_exog_list both exist precisely to exploit known-future inputs,
#        and both were handed an empty list. Half of what makes a TFT a TFT
#        was switched off.
#   NEW: compute them from the Date itself. A future date's day-of-week is
#        knowable today by definition, so there is no leak argument against
#        these -- they are the textbook known-future feature.
#        Sin/cos rather than raw integers so Friday and Monday are adjacent.
d = df["Date"].dt
df["dow_sin"]   = np.sin(2 * np.pi * d.dayofweek / 5)
df["dow_cos"]   = np.cos(2 * np.pi * d.dayofweek / 5)
df["month_sin"] = np.sin(2 * np.pi * (d.month - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (d.month - 1) / 12)
df["dom_norm"]  = (d.day - 1) / 30.0
df["is_month_end"]   = d.is_month_end.astype(float)
df["is_quarter_end"] = d.is_quarter_end.astype(float)
CALENDAR_COLS = ["dow_sin", "dow_cos", "month_sin", "month_cos",
                 "dom_norm", "is_month_end", "is_quarter_end"]

df["time_idx"] = df.groupby("group_id").cumcount()

# ── C15 ── the fold layout is checked before anything trains ──────────────
#   V6:  asserted only that each group had >= encoder + val + horizon rows.
#   NEW: the rolling-origin layout needs more, and finding that out 40 minutes
#        into a fit is expensive. Checked here with the arithmetic printed.
n_per_group = df.groupby("group_id").size()
need = INPUT_SIZE + N_FOLDS * TEST_SIZE + VAL_SIZE + HORIZON
assert n_per_group.min() >= need, (
    f"Shortest group has {n_per_group.min()} rows but the fold layout needs "
    f"{need}. Lower N_FOLDS or TEST_SIZE.")
print(f"   {n_per_group.min()} rows/group, target std {df['target'].std():.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5 — FEATURE LISTS                                  [CHANGES C16 … C20]
# ═══════════════════════════════════════════════════════════════════════════
# Every column is sorted by ONE question: at the moment I make the forecast,
# do I know this value for the day I am forecasting?
#     yes, always      -> KNOWN FUTURE   (calendar, scheduled releases)
#     only up to today -> HISTORICAL     (news, volatility, realised surprise)
#     it IS the answer -> not a feature  (target, ids, shifted duplicates)
# ═══════════════════════════════════════════════════════════════════════════
ID_COLS = {"Date", "group_id", "group", "time_idx", "target", "split",
           "leg1", "leg2", "leg3"}

# ── C16 ── *_lead1 columns dropped, for a different reason than V6's ──────
#   V6:  excluded macro_*_lead1 as a "leak guard", in the same category as
#        event_lead_1/2.
#   WHY: that was the wrong reason. A scheduled economic release is not a
#        leak -- the calendar publishes its date and importance weeks ahead.
#        V6 threw away the only legitimate known-future feature it had.
#   NEW: the *_lead1 columns are still dropped, but because they are
#        DUPLICATES: macro_leg1_lead1 at row t is just macro_leg1 at row t+1.
#        Keeping both hands the model the same number twice and invites an
#        off-by-one. The information is recovered properly below, by treating
#        the un-shifted scheduled columns as known-future.
LEAK_SUFFIXES = ("_lead1",)

numeric = [c for c in df.columns
           if c not in ID_COLS
           and pd.api.types.is_numeric_dtype(df[c])
           and not c.endswith(LEAK_SUFFIXES)]

# ── C17 ── known-future is no longer empty ────────────────────────────────
#   V6:  futr_exog_list = [] and time_varying_known_reals = [].
#   NEW: calendar (C14) plus the scheduled part of the macro calendar.
#        WHY these macro columns and not the others: macro_leg1 is
#        impact_score x direction, and impact_score comes from the release's
#        published `importance` -- known in advance. macro_surprise_* needs
#        the ACTUAL number, which is not known until the release lands, so it
#        stays historical. The _3d rolling sums are backward-looking, so they
#        stay historical too. This is the one place where being careless
#        would create a genuine leak, so the split is per-column, not
#        per-prefix.
KNOWN_MACRO = [c for c in numeric
               if c.startswith("macro_leg")
               and "surprise" not in c
               and not c.endswith("_3d")]
KNOWN_MACRO += [c for c in numeric if c.startswith("macro_n_events_")]

FUTR_COLS = CALENDAR_COLS + KNOWN_MACRO

# ── C18 ── regimes are labels, not magnitudes ─────────────────────────────
#   V6:  pd.factorize(df[col])[0].astype(float), fed into
#        time_varying_unknown_reals alongside continuous features.
#   WHY: factorize() assigns codes in order of first appearance, so the
#        numbers are arbitrary. Handing them to the model as floats says
#        "state 2 is twice state 1", which is meaningless for an HMM state.
#        (For regime_vol / regime_macro the order low/mid/high IS real, so
#        the old treatment was harmless there -- it was `regime` that was
#        being misread.)
#   NEW: strings for TFT (which has a proper categorical embedding path), and
#        an explicit one-hot copy for NeuralForecast, which takes numeric
#        exogenous only. One-hot is the correct encoding for unordered states.
CATEG_COLS = [c for c in ["regime", "regime_vol", "regime_macro"] if c in df.columns]
for c in CATEG_COLS:
    df[c] = df[c].round().astype(int).astype(str)

# ── C19 ── the single biggest feature change ──────────────────────────────
#   V6:  zscore and spread appear in NO feature list. Not gdelt_cols, not
#        dcc_garch_leg_cols, not macro_leg_cols, not calendar_cols.
#   WHY: the target is the next CHANGE in zscore. Mean reversion says that
#        change depends on how far the spread is currently stretched -- a
#        spring pulls back harder the further you pull it. Measured on your
#        sample, zscore correlates -0.095 with the target: the strongest
#        honest relationship in the whole file, and the only one with an
#        economic story behind it. Every other feature was below 0.07.
#        Today's z-score is known today, so including it is not leakage.
#        The models were being asked to predict the spring's next move
#        without being told how far it is stretched.
#   NEW: included, and asserted at the bottom of this cell so it can never
#        quietly fall out again.
PRICE_STATE = [c for c in ["zscore", "spread", "rho_mean", "rho_min"] if c in df.columns]

REGIME_OH = []
for c in CATEG_COLS:
    for lvl in sorted(df[c].unique()):
        name = f"{c}_is{lvl}"
        df[name] = (df[c] == lvl).astype(float)
        REGIME_OH.append(name)

HIST_COLS = [c for c in numeric
             if c not in set(FUTR_COLS) and c not in set(CATEG_COLS)]

# ── C20 ── constant and NaN columns are dropped, not fed ──────────────────
#   V6:  event_flag, event_lag_1 and event_lag_2 were all wired into both
#        models as features.
#   WHY: per the data-pipeline review, all three are constant -- the old
#        event_flag compared abs(x) against a percentile of the SIGNED
#        series, and since shock_events_diff is negative every day, the test
#        was always true. V6 was feeding three columns of pure nothing to
#        every model, and they still showed a 0.26 correlation with the
#        target because of two rows at the very start.
#   NEW: anything constant or NaN-bearing is dropped here with its name
#        printed, whatever produced it. This is a safety net, not a fix --
#        the real fix is upstream in the data pipeline.
def _usable(cols):
    keep, dropped = [], []
    for c in cols:
        s = df[c]
        if s.isna().any() or s.nunique(dropna=False) <= 1:
            dropped.append(c)
        else:
            keep.append(c)
    return keep, dropped

HIST_COLS, dropped_h = _usable(HIST_COLS)
FUTR_COLS, dropped_f = _usable(FUTR_COLS)

print(f"known-future : {len(FUTR_COLS):3d}  ({len(CALENDAR_COLS)} calendar + "
      f"{len(FUTR_COLS)-len(CALENDAR_COLS)} scheduled macro)      [V6 had 0]")
print(f"historical   : {len(HIST_COLS):3d}  (includes {PRICE_STATE})")
print(f"categorical  : {len(CATEG_COLS):3d}  {CATEG_COLS}")
if dropped_h or dropped_f:
    print(f"dropped {len(dropped_h)+len(dropped_f)} constant/NaN columns: "
          f"{(dropped_h+dropped_f)[:8]}{' ...' if len(dropped_h+dropped_f) > 8 else ''}")
assert all(c in HIST_COLS for c in PRICE_STATE), \
    "zscore/spread did not survive the usability filter — check the dataset."
HIST_NF = HIST_COLS + REGIME_OH      # what NeuralForecast gets (C18)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6 — ROLLING-ORIGIN FOLDS                            [CHANGE C21 — new cell]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C21 ── one 5-day test window -> N walked-forward windows ──────────────
#   V6:  assign_split() marked the last 5 rows of each group as test.
#            train (55,540 rows)  2013-03-05 -> 2023-10-25
#            val    (1,200 rows)  2023-10-26 -> 2024-01-17
#            test     (100 rows)  2024-01-18 -> 2024-01-24     <-- one week
#   WHY: 100 rows cannot rank five architectures. On your data the target has
#        sd 0.153 and absolute errors have sd 0.105, which puts the noise on
#        a 100-row MAE at +/-9.6% treating rows as independent -- and they are
#        not, because five consecutive days inside one group move together.
#        Treating the 20 groups as the independent unit gives +/-21%. Every
#        model in V6's leaderboard sat inside that band.
#        The other problem: those five days are ONE market week. Whatever
#        happened that week decided the ranking.
#   NEW: N_FOLDS windows of TEST_SIZE days, walked forward. Each fold trains
#        only on rows before its own validation window, so no fold ever sees
#        its own future. With the defaults: 3 x 250 x n_groups test rows,
#        spread across three different market periods, which is enough to
#        rank on AND enough to put an error bar on each number.
#
#   COST: three folds means training each model three times. That is exactly
#         why the resume system in cell 3 exists -- the run is now allowed to
#         take more than one session.
# ═══════════════════════════════════════════════════════════════════════════
n_rows = int(df.groupby("group_id").size().min())

FOLDS = []
for k in range(N_FOLDS):
    test_end   = n_rows - (N_FOLDS - 1 - k) * TEST_SIZE
    test_start = test_end - TEST_SIZE
    val_start  = test_start - VAL_SIZE
    FOLDS.append({"fold": k,
                  "train_end": val_start,      # exclusive
                  "val_start": val_start, "val_end": test_start,
                  "test_start": test_start, "test_end": test_end})

def tag_fold(frame, f):
    """Label rows train/val/test for one fold; drop anything after the test."""
    g = frame.copy()
    g["split"] = "drop"
    g.loc[g["time_idx"] <  f["train_end"], "split"] = "train"
    g.loc[(g["time_idx"] >= f["val_start"])  & (g["time_idx"] < f["val_end"]),  "split"] = "val"
    g.loc[(g["time_idx"] >= f["test_start"]) & (g["time_idx"] < f["test_end"]), "split"] = "test"
    return g[g["split"] != "drop"].reset_index(drop=True)

print(f"{N_FOLDS} rolling-origin folds, {df['group_id'].nunique()} groups\n")
for f in FOLDS:
    s = tag_fold(df, f)
    dts = {k: (v["Date"].min().date(), v["Date"].max().date())
           for k, v in s.groupby("split")}
    print(f"  fold {f['fold']}: train {f['train_end']:>5} rows/grp "
          f"({dts['train'][0]} -> {dts['train'][1]})   "
          f"val {dts['val'][0]}->{dts['val'][1]}   "
          f"TEST {dts['test'][0]}->{dts['test'][1]}  "
          f"({(s['split']=='test').sum():,} scorable rows)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7 — METRICS AND BASELINES                        [CHANGES C22, C23, C24]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C22 ── baselines, which V6 had none of ────────────────────────────────
#   V6:  the leaderboard held TFT, NBEATSx, NHITS, PatchTST and
#        Spacetimeformer, sorted by MAE. Nothing else.
#   WHY: that table can tell you which network is least bad. It cannot tell
#        you whether ANY of them is worth running. Measured on your data:
#            MAE, always predict 0                       0.11035
#            MAE, best linear fit on zscore (in-sample)  0.10995   <- 0.4% better
#            MAE, predict yesterday's change             0.15518   <- 41% worse
#        Predicting a flat zero scores 0.110. The single best rule anyone
#        could fit, using the answer key, scores 0.110. Until a model beats
#        0.110 by more than the confidence band, "TFT won" means nothing.
#   NEW: three baselines, fitted on train only, scored on exactly the same
#        rows as the models, in every fold. They cost seconds and they are
#        the only thing that makes the other numbers interpretable.
#
# ── C23 ── the confidence interval resamples GROUPS, not rows ─────────────
#   V6:  a single MAE number per model, no interval.
#   WHY: rows are not independent. Days inside one group share a spread, a
#        volatility regime and a news stream. Bootstrapping rows would give a
#        confidently narrow interval that is simply wrong. Resampling whole
#        groups respects the correlation and gives an honest width.
#
# ── C24 ── directional accuracy added ─────────────────────────────────────
#   V6:  MAE only.
#   WHY: for a trading rule, being right about the SIGN matters more than
#        being close in magnitude -- you size the trade separately. Measured
#        on the biggest 75% of moves, because the sign of a near-zero move is
#        noise and would just dilute the statistic. 0.50 is a coin flip.
# ═══════════════════════════════════════════════════════════════════════════
def metrics(y_true, y_pred, groups, n_boot=1000, seed=SEED):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_pred - y_true
    mae, rmse = np.abs(err).mean(), np.sqrt((err ** 2).mean())

    # directional accuracy on the moves that are big enough to trade
    big = np.abs(y_true) > np.percentile(np.abs(y_true), 25)
    if not big.any() or np.allclose(y_pred, 0):
        dir_acc = np.nan          # a constant-zero forecast has no direction
    else:
        dir_acc = float((np.sign(y_pred[big]) == np.sign(y_true[big])).mean())

    # block bootstrap over groups
    rng = np.random.default_rng(seed)
    gs = np.asarray(groups)
    uniq = np.unique(gs)
    idx_by_g = {g: np.where(gs == g)[0] for g in uniq}
    boots = []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        sel = np.concatenate([idx_by_g[g] for g in pick])
        boots.append(np.abs(err[sel]).mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])

    return {"mae": float(mae), "mae_lo": float(lo), "mae_hi": float(hi),
            "rmse": float(rmse), "dir_acc": dir_acc, "n": int(len(y_true))}


def run_baselines(fold_df):
    """Three reference points, fitted on train only, scored on test."""
    tr = fold_df[fold_df["split"] == "train"]
    te = fold_df[fold_df["split"] == "test"]
    out = {}

    # 1. the null model
    out["Baseline-zero"] = np.zeros(len(te))

    # 2. each group's own average change during training
    #    catches a model that is only learning a per-group intercept
    gmean = tr.groupby("group_id")["target"].mean()
    out["Baseline-groupmean"] = te["group_id"].map(gmean).fillna(0.0).values

    # 3. mean reversion: target ~ a * zscore + b, fitted per group on train.
    #    This is the economic story the whole strategy rests on, written as
    #    two parameters. If no neural net beats it, the nets are not adding
    #    anything and the answer is better features, not more epochs.
    preds = np.zeros(len(te))
    te_idx = {g: np.where(te["group_id"].values == g)[0] for g in te["group_id"].unique()}
    for g, rows in te_idx.items():
        t = tr[tr["group_id"] == g]
        if len(t) > 50 and t["zscore"].std() > 0:
            a, b = np.polyfit(t["zscore"].values, t["target"].values, 1)
            preds[rows] = a * te["zscore"].values[rows] + b
    out["Baseline-meanrev"] = preds

    return {name: metrics(te["target"].values, p, te["group_id"].values)
            for name, p in out.items()}


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 8 — GPU SETUP                                     [CHANGES C25, C26]
# ═══════════════════════════════════════════════════════════════════════════
import torch

# ── C25 ── the accelerator is stated, not inferred ────────────────────────
#   V6:  pl.Trainer(max_epochs=50, callbacks=[...], logger=..., 
#                   gradient_clip_val=0.1)
#        No accelerator, no devices, no precision. Lightning auto-detects,
#        which is fine when a GPU is attached and silent when it is not.
#   WHY: 847 batches/epoch at 0.52 it/s in your saved output. For a TFT with
#        hidden_size=32 and batch 64 that is CPU throughput. 27 min/epoch x 50
#        epochs = 20+ hours, on a platform that stops you at 9. The run could
#        not have finished no matter how long you left it open.
#   NEW: accelerator and precision are set explicitly and printed, so "is it
#        on the GPU" is answered before training rather than inferred from
#        how slow it feels an hour in.
#        fp16 roughly halves memory and gives the T4 its tensor cores; with
#        batch 128 instead of 64 the GPU stops waiting on the dataloader.
USE_GPU = torch.cuda.is_available()
ACCEL   = "gpu" if USE_GPU else "cpu"
PREC    = PRECISION if USE_GPU else "32-true"     # fp16 is meaningless on CPU

if USE_GPU:
    torch.set_float32_matmul_precision("medium")  # allow TF32 matmul paths
    print(f"✅ {torch.cuda.get_device_name(0)}  |  precision={PREC}  batch={BATCH_SIZE}")
else:
    print("⚠️  NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart. "
          "Training on CPU will not finish inside a session.")

# ── C26 ── every fit carries a wall clock ─────────────────────────────────
#   V6:  max_epochs=50 with EarlyStopping(patience=5) and no time bound. The
#        only thing that could end the run was the platform killing it.
#   NEW: max_time caps a single model at PER_UNIT_BUDGET_MIN minutes. It
#        stops, keeps its best checkpoint, gets scored and synced. A model
#        that turns out to be slow costs you one unit, not the session.
#        devices=1 on purpose: Kaggle's T4 x2 would trigger DDP, which is
#        awkward inside a notebook and gains little at this model size.
TRAINER_KW = dict(
    accelerator=ACCEL,
    devices=1,
    precision=PREC,
    max_time={"minutes": PER_UNIT_BUDGET_MIN},
    enable_progress_bar=True,
    logger=False,
)

def time_left():
    return SESSION_DEADLINE - time.time()

def time_left_str():
    m = int(max(0, time_left()) // 60)
    return f"{m//60}h{m%60:02d}m"


## Step 4 — Model units

One function per model family. Neither trains anything yet — the loop below calls them.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 9 — TFT UNIT                                      [CHANGES C27 … C31]
# ═══════════════════════════════════════════════════════════════════════════
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint


def train_tft(fold_df, f, unit_dir):
    tr_va = fold_df[fold_df["split"].isin(["train", "val"])]

    training = TimeSeriesDataSet(
        tr_va[tr_va["time_idx"] < f["val_start"]],
        time_idx="time_idx", target="target", group_ids=["group_id"],
        max_encoder_length=INPUT_SIZE, max_prediction_length=HORIZON,

        # ── C27 ── known_reals is no longer empty (see C14, C17) ──────────
        #   V6: time_varying_known_reals=calendar_cols, and calendar_cols
        #       was []. TFT's entire known-future pathway was unused.
        time_varying_known_reals=FUTR_COLS,

        # ── C28 ── zscore and spread are in here now (see C19) ────────────
        #   V6: ["target","event_lag_1","event_lag_2"] + event_flag +
        #       gdelt + dcc_garch_leg + macro_leg + regime + macro_pressure.
        #       Three of those were constant columns and the most predictive
        #       column in the dataset was absent.
        time_varying_unknown_reals=["target"] + HIST_COLS,

        # ── C29 ── regimes as categoricals (see C18) ──────────────────────
        #   V6: fed in as factorized floats among the continuous reals.
        time_varying_unknown_categoricals=CATEG_COLS,

        #   NEW: group identity as a static categorical. V6 used group_id only
        #   for the normalizer, so the model could not learn that one trio
        #   behaves differently from another.
        static_categoricals=["group_id"],

        target_normalizer=GroupNormalizer(groups=["group_id"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=True,
    )

    # ── C30 ── the validation set is ~1,100 windows, not 20 ───────────────
    #   V6:  validation_dataset = TimeSeriesDataSet.from_dataset(
    #            training_dataset, train_val_df, predict=True, ...)
    #        `predict=True` yields exactly ONE window per series. With 20
    #        groups that is 20 samples.
    #   WHY: val_loss, ModelCheckpoint(monitor="val_loss") and
    #        EarlyStopping(patience=5) were ALL decided by those 20 samples.
    #        Which checkpoint got called "best" was close to arbitrary, and
    #        no hyperparameter comparison run on top of it was measuring
    #        what it looked like it was measuring.
    #   NEW: min_prediction_idx=val_start instead, which generates every
    #        window whose prediction step falls inside the 120-day validation
    #        range -- roughly 120 windows per group.
    validation = TimeSeriesDataSet.from_dataset(
        training, tr_va, min_prediction_idx=f["val_start"], stop_randomization=True)

    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=2,
                 persistent_workers=True, pin_memory=USE_GPU)
    # persistent_workers + pin_memory: V6 respawned its 2 workers every epoch
    # and copied through pageable memory, which on Kaggle's 4 vCPUs is a real
    # share of the per-epoch time.
    train_loader = training.to_dataloader(train=True, **dl_kw)
    val_loader   = validation.to_dataloader(train=False, **dl_kw)

    ckpt = ModelCheckpoint(dirpath=f"{unit_dir}/ckpt", filename="best",
                           monitor="val_loss", mode="min", save_top_k=1)
    model = TemporalFusionTransformer.from_dataset(
        training,
        # lr 1e-3 -> 3e-3 and dropout 0.1 -> 0.15: the wall clock now caps
        # training, so it has to converge faster, and the extra dropout
        # offsets the larger effective feature set from C19/C27.
        learning_rate=3e-3, hidden_size=32, attention_head_size=4,
        dropout=0.15, hidden_continuous_size=16, loss=QuantileLoss(),
        log_interval=-1, optimizer="adam")

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, gradient_clip_val=0.1,
        callbacks=[ckpt, EarlyStopping(monitor="val_loss", patience=4, mode="min")],
        **TRAINER_KW)
    trainer.fit(model, train_loader, val_loader)

    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)

    # ── C31 ── the test window is rolled through, not sampled once ────────
    #   V6:  test_dataset = from_dataset(training_dataset, tft_df,
    #                                    predict=True, ...)
    #        then trainer.test(...). `predict=True` again means one window
    #        per group: 20 sequences x 5 steps = 100 numbers, and the score
    #        came out of Lightning's logged metrics via a substring search
    #        for any key containing "mae" -- one library version away from
    #        silently reporting a different metric.
    #   NEW: min_prediction_idx=test_start generates a window for every day
    #        in the test range, which is a proper rolling one-step-ahead
    #        forecast. Predictions are merged back on (group_id, time_idx)
    #        so every number is matched to its own real target, and the MAE
    #        is computed here in units we control rather than read out of a
    #        logger dictionary.
    test_ds = TimeSeriesDataSet.from_dataset(
        training, fold_df, min_prediction_idx=f["test_start"], stop_randomization=True)
    test_dl = test_ds.to_dataloader(train=False, batch_size=BATCH_SIZE,
                                    num_workers=2, pin_memory=USE_GPU)
    out = best.predict(test_dl, mode="prediction", return_index=True)
    preds = np.asarray(out.output).reshape(len(out.index), -1)[:, 0]

    scored = out.index[["group_id", "time_idx"]].copy()
    scored["pred"] = preds
    scored = scored.merge(fold_df[["group_id", "time_idx", "target", "Date"]],
                          on=["group_id", "time_idx"], how="inner")
    return scored, {"val_loss": float(ckpt.best_model_score),
                    "epochs": int(trainer.current_epoch)}


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 10 — NEURALFORECAST UNIT (NBEATSx / NHITS / PatchTST)  [CHANGES C32 … C38]
# ═══════════════════════════════════════════════════════════════════════════
# V6's Cell I could not reach a single training batch. Three separate
# construction-time exceptions, all fixed below, plus three design changes.
# ═══════════════════════════════════════════════════════════════════════════
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, NHITS, PatchTST
from neuralforecast.losses.pytorch import MQLoss

QUANTILES = [0.1, 0.5, 0.9]

# ── C32 ── Trainer settings are flat keyword arguments ────────────────────
#   V6:  trainer_kwargs={"logger": CSVLogger(save_dir=..., name="logs", ...)}
#   WHY: NeuralForecast models take Lightning Trainer settings as
#        **trainer_kwargs -- varargs, not a parameter called trainer_kwargs.
#        Passing a dict by that name puts the literal key "trainer_kwargs"
#        into the dict that is later splatted into pl.Trainer, which raises:
#            TypeError: Trainer.__init__() got an unexpected keyword
#                       argument 'trainer_kwargs'
#        Verified against neuralforecast 3.2.1.
#   NEW: passed flat, along with the GPU/precision/wall-clock settings from
#        C25/C26 so the NF models get the same treatment as TFT.
TRAINER_NF = dict(
    accelerator=ACCEL, devices=1, precision=PREC,
    max_time={"minutes": PER_UNIT_BUDGET_MIN},
    logger=False, enable_progress_bar=True,
    enable_model_summary=False, enable_checkpointing=False,
)
# Lightning prints "Ignoring Trainer(max_time=...), callbacks list already
# contains a Timer" when it builds the second (prediction) trainer. Harmless
# -- the training trainer honoured the budget; the message is about the other.


def _build(name):
    common = dict(
        h=HORIZON, input_size=INPUT_SIZE,

        # ── C33 ── valid_loss must match the loss family ──────────────────
        #   V6:  loss=MQLoss(quantiles=[0.1,0.5,0.9]), valid_loss=MAE()
        #   WHY: neuralforecast raises at construction:
        #            Exception: Please set valid_loss to MQLoss() or
        #                       HuberMQLoss() when training with MQLoss
        #        NBEATSx is the FIRST model built in V6's Cell I, so this
        #        alone killed the cell before PatchTST was even reached.
        loss=MQLoss(quantiles=QUANTILES),
        valid_loss=MQLoss(quantiles=QUANTILES),

        # ── C34 ── a step budget the models can learn inside ──────────────
        #   V6:  max_steps=300, val_check_steps=50,
        #        early_stop_patience_steps=5.
        #   WHY: 300 steps at the default batch is a small fraction of one
        #        pass over 55,540 rows -- those models were barely trained.
        #        And patience is counted in VALIDATION CHECKS: 5 checks x 50
        #        steps = 250, so early stopping could essentially never fire
        #        inside a 300-step budget. The names suggested one thing and
        #        the arithmetic did another.
        #   NEW: a real budget, checked often enough for patience to mean
        #        something, and bounded by the wall clock rather than a
        #        guessed step count.
        max_steps=MAX_STEPS, val_check_steps=200,
        early_stop_patience_steps=6,

        # ── C35 ── robust scaling instead of standard ─────────────────────
        #   V6:  scaler_type="standard"
        #   WHY: these features are heavy-tailed by construction -- news
        #        shock counts, GARCH volatilities, macro surprises clipped at
        #        +/-10. A mean/std scaler lets a handful of crisis days set
        #        the scale for the whole column. Median/IQR does not.
        scaler_type="robust",

        batch_size=32, windows_batch_size=BATCH_SIZE,
        random_seed=SEED, alias=name,
        **TRAINER_NF,
    )

    # ── C36 ── PatchTST is the univariate control ─────────────────────────
    #   V6:  PatchTST(patch_len=6, stride=1, d_model=128, n_heads=4,
    #                 e_layers=3, dropout=0.1, **common_kwargs)
    #        with common_kwargs carrying hist_exog_list (94 columns).
    #   WHY: TWO failures here.
    #        (a) PatchTST declares EXOGENOUS_HIST = False and the base class
    #            raises rather than ignoring the request --
    #            neuralforecast/common/_base_model.py line 269:
    #              if not self.EXOGENOUS_HIST and self.hist_exog_size > 0:
    #                  raise Exception(f"{name} does not support historical
    #                                   exogenous variables.")
    #        (b) `d_model` and `e_layers` are not parameters of this version.
    #            Unknown kwargs fall through to pl.Trainer and raise TypeError.
    #            The real names are `hidden_size` and `encoder_layers`.
    #   NEW: no exogenous at all, correct parameter names, and kept
    #        deliberately as the control: "how far does the target's own
    #        history get you with no extra data?" If the exogenous models
    #        cannot beat it, the whole feature-engineering pipeline is not
    #        earning its keep -- which is a genuinely useful thing to learn.
    if name == "PatchTST":
        return PatchTST(patch_len=8, stride=4, hidden_size=64, n_heads=4,
                        encoder_layers=2, dropout=0.1, **common)

    common.update(hist_exog_list=HIST_NF, futr_exog_list=FUTR_COLS)
    if name == "NBEATSx":
        return NBEATSx(stack_types=["identity", "trend", "seasonality"],
                       n_blocks=[2, 2, 2], mlp_units=[[256, 256]] * 3,
                       dropout_prob_theta=0.1, **common)
    if name == "NHITS":
        return NHITS(n_freq_downsample=[4, 2, 1], n_blocks=[2, 2, 2],
                     mlp_units=[[256, 256]] * 3, dropout_prob_theta=0.1, **common)
    raise ValueError(name)


def train_nf(fold_df, f, unit_dir, name):
    # ── C37 ── ds is the integer time index, with freq=1 ──────────────────
    #   V6:  nf_df["ds"] = the Date, and NeuralForecast(models=[...], freq="B")
    #   WHY: freq="B" tells NeuralForecast the series is on an unbroken
    #        business-day index. Your dates are TRADING days -- every market
    #        holiday is a missing business day. NF then reindexes onto days
    #        that do not exist in the data, which either errors or quietly
    #        inserts rows. Using the integer time_idx removes the calendar
    #        from the problem entirely; the real dates are merged back after.
    use = ["group_id", "time_idx", "target"] + HIST_NF + FUTR_COLS
    nf_df = (fold_df[list(dict.fromkeys(use))]
             .rename(columns={"group_id": "unique_id",
                              "time_idx": "ds", "target": "y"}))

    nf = NeuralForecast(models=[_build(name)], freq=1)

    # ── C38 ── cross_validation instead of a single predict() ─────────────
    #   V6:  nf.fit(nf_train, val_size=val_window)  then  nf.predict()
    #   WHY: predict() returns the h steps immediately after the training
    #        data ends -- with h=5 that is five rows per group, which is
    #        exactly where V6's 100-row test set came from.
    #   NEW: cross_validation with step_size=1 fits ONCE (refit=False) and
    #        then rolls a one-step forecast across the whole test window,
    #        producing TEST_SIZE scored days per group. Same trained model,
    #        250x the evidence.
    cv = nf.cross_validation(nf_df, val_size=VAL_SIZE, n_windows=TEST_SIZE,
                             step_size=1, refit=False)

    med = f"{name}-median"
    assert med in cv.columns, f"no median column for {name}: {list(cv.columns)}"
    # V6 had a pick_quantile_cols() helper that guessed at these names. The
    # assert is the same idea, one line, and it names the columns it did find.

    nf.save(unit_dir, overwrite=True)

    scored = cv.rename(columns={"unique_id": "group_id", "ds": "time_idx",
                                med: "pred", "y": "target"})
    scored = scored[["group_id", "time_idx", "pred", "target"]]
    scored = scored.merge(fold_df[["group_id", "time_idx", "Date"]],
                          on=["group_id", "time_idx"], how="left")
    lo, hi = f"{name}-lo-80.0", f"{name}-hi-80.0"
    if lo in cv.columns:
        scored["pred_lo"] = cv[lo].values
        scored["pred_hi"] = cv[hi].values
    return scored, {"max_steps": MAX_STEPS}


## Step 5 — Run

This is the resumable loop. Run it, let it go as long as the session lasts, and if the
session dies just run the notebook again from the top — it continues from the last
finished unit.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 11 — RESUMABLE MAIN LOOP                         [CHANGES C39, C40, C41]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C39 ── the run is a list of units, with a ledger ──────────────────────
#   V6:  Cells C, I and J each trained their models top to bottom with no
#        record of what had completed. Restarting the notebook restarted
#        everything -- and Cell 16 deleted the previous outputs first.
#   NEW: one unit per (fold, model). For each one:
#            already done?  -> skip and load its saved score
#            no time left?  -> stop cleanly, sync, tell you to rerun
#            otherwise      -> train, score, save, mark done, SYNC
#        Because both the ledger and the scores live inside the synced
#        snapshot, a restart after a kill picks up at the exact unit that was
#        interrupted.
#
# ── C40 ── each unit is isolated ──────────────────────────────────────────
#   V6:  Cell I built NBEATSx, NHITS and PatchTST at the top of one cell.
#        The PatchTST exception therefore killed all three -- two models that
#        would have trained fine never got the chance. Downstream, Cell M/N/O
#        referenced nbeats_preds and friends, so every later cell failed too.
#   NEW: try/except per unit. A crash is written to STATE["failed"] with its
#        exception text and the loop moves on. One broken library cannot take
#        the session down, and the failure is still visible afterwards
#        instead of scrolling past in a traceback.
#
# ── C41 ── baselines run first ────────────────────────────────────────────
#   They are cheap, they cannot fail, and they define what "good" means for
#   everything that follows. Running them first means that even a session
#   that dies early leaves you with something interpretable.
# ═══════════════════════════════════════════════════════════════════════════
import traceback

RESULTS = {}          # unit_key -> metrics dict


def unit_key(fold, model):
    return f"f{fold}_{model}"


def load_done_unit(key):
    p = f"{RUN_DIR}/{key}/metrics.json"
    return json.load(open(p)) if os.path.exists(p) else None


# ── baselines first: cheap, and they define what "good" means ──────────────
for f in FOLDS:
    key = unit_key(f["fold"], "baselines")
    if key in STATE["done"]:
        RESULTS.update(load_done_unit(key) or {})
        continue
    fold_df = tag_fold(df, f)
    res = {f"{k}|f{f['fold']}": v for k, v in run_baselines(fold_df).items()}
    os.makedirs(f"{RUN_DIR}/{key}", exist_ok=True)
    json.dump(res, open(f"{RUN_DIR}/{key}/metrics.json", "w"), indent=2)
    STATE["done"][key] = time.time(); save_state(STATE)
    RESULTS.update(res)
    print(f"fold {f['fold']} baselines  "
          + "  ".join(f"{k.split('|')[0].replace('Baseline-','')}={v['mae']:.5f}"
                      for k, v in res.items()))
sync_out("baselines")

# ── then the models ────────────────────────────────────────────────────────
print(f"\n{'='*70}\nUNITS: {len(FOLDS)} folds x {len(MODELS)} models = "
      f"{len(FOLDS)*len(MODELS)}   |   session budget left {time_left_str()}\n{'='*70}")

stopped_early = False
for f in FOLDS:
    for model in MODELS:
        key = unit_key(f["fold"], model)

        if key in STATE["done"]:
            m = load_done_unit(key)
            if m:
                RESULTS.update(m)
                print(f"✓ {key:22s} already done   MAE {list(m.values())[0]['mae']:.5f}")
            continue
        if key in STATE["failed"]:
            print(f"✗ {key:22s} failed before — delete it from run_state.json to retry")
            continue
        # refuse to start something the session cannot finish (C6/C26):
        # 55 min of training budget + 10 min of headroom for scoring + sync
        if time_left() < PER_UNIT_BUDGET_MIN * 60 + 600:
            print(f"\n⏳ Not enough session time for {key} "
                  f"({time_left_str()} left). Stopping cleanly.")
            stopped_early = True
            break

        print(f"\n▶ {key}   ({time_left_str()} of session left)")
        t0 = time.time()
        unit_dir = f"{RUN_DIR}/{key}"
        os.makedirs(unit_dir, exist_ok=True)
        try:
            fold_df = tag_fold(df, f)
            if model == "TFT":
                scored, extra = train_tft(fold_df, f, unit_dir)
            else:
                scored, extra = train_nf(fold_df, f, unit_dir, model)

            scored = scored.dropna(subset=["pred", "target"])
            m = metrics(scored["target"].values, scored["pred"].values,
                        scored["group_id"].values)
            m.update(extra); m["minutes"] = round((time.time() - t0) / 60, 1)
            res = {f"{model}|f{f['fold']}": m}

            scored.to_parquet(f"{unit_dir}/test_predictions.parquet", index=False)
            json.dump(res, open(f"{unit_dir}/metrics.json", "w"), indent=2)
            STATE["done"][key] = time.time(); save_state(STATE)
            RESULTS.update(res)
            print(f"  ✅ MAE {m['mae']:.5f} [{m['mae_lo']:.5f}, {m['mae_hi']:.5f}]  "
                  f"dir {m['dir_acc']:.3f}  n={m['n']:,}  {m['minutes']}min")
        except Exception as e:
            STATE["failed"][key] = f"{type(e).__name__}: {e}"
            save_state(STATE)
            print(f"  ❌ {type(e).__name__}: {e}")
            traceback.print_exc(limit=3)
        sync_out(key)          # ← the line that makes a dead session cheap
    if stopped_early:
        break

print(f"\n{'done' if not stopped_early else 'PAUSED — rerun this notebook to continue'}"
      f"  |  {len(STATE['done'])} units complete, {len(STATE['failed'])} failed")


## Step 6 — Results

---

### If it stops before finishing

Run the notebook again from the top with the **same `RUN_NAME`**. It will restore the
snapshot, print `units already complete: N`, and pick up at the next unit.

### To retry a failed unit

Failed units are listed in `run_state.json` under `failed` and are skipped on later runs
so a broken model cannot burn a whole session. To retry one, delete its entry:

```python
del STATE["failed"]["f1_TFT"]
save_state(STATE)
sync_out("cleared failure")
```

### To tune a model without redoing everything

Delete only that model's units from `STATE["done"]`, sync, and rerun. Every other unit
stays cached.

### If a run is genuinely finished and you want a clean one

Change `RUN_NAME` in the config cell. The old snapshot is untouched, so you can always
go back to it.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 12 — LEADERBOARD                                 [CHANGES C42, C43]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C42 ── a missing model can no longer vanish silently ──────────────────
#   V6:  leaderboard = pd.DataFrame(rows).dropna(subset=["test_mae"])
#        Spacetimeformer never ran, so stf_test_mae stayed None, so dropna()
#        removed the row and the printed table looked complete. The ensemble
#        was computed in Cell K, saved, and then never scored or added at all.
#   NEW: the table is built from RESULTS, which only contains units that
#        actually produced predictions, and the run_state ledger separately
#        records anything that failed. What is absent is absent on purpose
#        and named in the manifest.
#
# ── C43 ── skill vs zero, and interval overlap, are printed as conclusions ─
#   V6:  a sorted MAE column. Reading it required knowing the noise floor,
#        which was nowhere on the page.
#   NEW: skill_vs_zero and the bootstrap bounds are computed, and the
#        notebook states in words whether the winner is actually
#        distinguishable from predicting nothing. A leaderboard should not
#        need the reader to supply the statistics.
# ═══════════════════════════════════════════════════════════════════════════
# Read this table in one specific order:
#   1. Look at `skill_vs_zero` first. Negative means the model is worse than
#      predicting nothing at all. Most FX models land here; it is not a
#      disaster, it is information.
#   2. Then look at whether the confidence intervals overlap. Two models
#      whose [mae_lo, mae_hi] ranges overlap are not distinguishable, no
#      matter how the rows are sorted.
#   3. `dir_acc` is accuracy of the predicted SIGN on the biggest 75% of
#      moves. For a trading rule this matters more than MAE — 0.50 is a coin
#      flip, and anything at 0.53+ that holds across folds is real.
# ═══════════════════════════════════════════════════════════════════════════
rows = []
for k, m in RESULTS.items():
    model, fold = k.split("|")
    rows.append({"model": model, "fold": fold, **m})
per_fold = pd.DataFrame(rows)

zero = (per_fold[per_fold["model"] == "Baseline-zero"]
        .set_index("fold")["mae"].to_dict())
per_fold["skill_vs_zero"] = per_fold.apply(
    lambda r: 1 - r["mae"] / zero.get(r["fold"], np.nan), axis=1)

board = (per_fold.groupby("model")
         .agg(mae=("mae", "mean"), mae_lo=("mae_lo", "mean"), mae_hi=("mae_hi", "mean"),
              rmse=("rmse", "mean"), dir_acc=("dir_acc", "mean"),
              skill_vs_zero=("skill_vs_zero", "mean"),
              folds=("fold", "nunique"), n=("n", "sum"))
         .sort_values("mae").reset_index())
board["skill_vs_zero"] = (board["skill_vs_zero"] * 100).round(2)

print("\n" + "=" * 92)
print(" LEADERBOARD — mean across folds, MAE bounds are 95% bootstrap over groups")
print("=" * 92)
print(board.to_string(index=False, float_format=lambda v: f"{v:.5f}"))

best = board.iloc[0]
zb = board[board["model"] == "Baseline-zero"]
if len(zb):
    zb = zb.iloc[0]
    if best["mae"] >= zb["mae"]:
        print("\n⚠️  Nothing beats predicting zero. The features carry no usable "
              "one-day signal as currently built — that is a real result, and it "
              "means the next move is more/better features, not more epochs.")
    elif best["mae_hi"] > zb["mae_lo"]:
        print(f"\n⚠️  {best['model']} is ahead of predict-zero but their confidence "
              "intervals overlap. Treat the gap as unproven until it survives "
              "more folds.")
    else:
        print(f"\n✅ {best['model']} beats predict-zero with non-overlapping "
              f"intervals: {best['mae']:.5f} vs {zb['mae']:.5f} "
              f"({best['skill_vs_zero']:.1f}% better).")

print("\nPer fold:")
print(per_fold[["model", "fold", "mae", "dir_acc", "n"]]
      .sort_values(["fold", "mae"]).to_string(index=False, float_format=lambda v: f"{v:.5f}"))

os.makedirs(f"{RUN_DIR}/leaderboard", exist_ok=True)
board.to_csv(f"{RUN_DIR}/leaderboard/leaderboard.csv", index=False)
per_fold.to_csv(f"{RUN_DIR}/leaderboard/per_fold.csv", index=False)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 13 — MANIFEST + FINAL SYNC                            [CHANGE C44]
# ═══════════════════════════════════════════════════════════════════════════
#
# ── C44 ── the manifest records the FEATURE LISTS too ─────────────────────
#   V6:  Cell Q wrote a manifest of file paths and hyperparameters. Good, and
#        kept here.
#   NEW: it also records exactly which columns went in as known-future,
#        historical and categorical, plus which units succeeded and which
#        failed. Six weeks from now, "why did that run score differently"
#        is answerable from this file alone instead of from memory.
# ═══════════════════════════════════════════════════════════════════════════
manifest = {
    "run": RUN_NAME,
    "finished_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "data": {"path": DATA_PATH, "rows": int(len(df)),
             "groups": int(df["group_id"].nunique()),
             "date_min": str(df["Date"].min().date()),
             "date_max": str(df["Date"].max().date())},
    "setup": {"horizon": HORIZON, "input_size": INPUT_SIZE, "n_folds": N_FOLDS,
              "test_size": TEST_SIZE, "val_size": VAL_SIZE, "seed": SEED,
              "batch_size": BATCH_SIZE, "precision": PREC, "accelerator": ACCEL},
    "features": {"known_future": FUTR_COLS, "historical": HIST_COLS,
                 "categorical": CATEG_COLS,
                 "n_known": len(FUTR_COLS), "n_hist": len(HIST_COLS)},
    "units_done": sorted(STATE["done"]),
    "units_failed": STATE["failed"],
    "leaderboard": board.to_dict(orient="records"),
}
json.dump(manifest, open(f"{RUN_DIR}/manifest.json", "w"), indent=2, default=str)

sync_out("final")

# a plain copy of the results in Drive, separate from the resume snapshot,
# so you can open them without unzipping anything
if USE_DRIVE and DRIVE and DRIVE.ok:
    for p, n in [(f"{RUN_DIR}/leaderboard/leaderboard.csv", f"{RUN_NAME}_leaderboard.csv"),
                 (f"{RUN_DIR}/leaderboard/per_fold.csv",   f"{RUN_NAME}_per_fold.csv"),
                 (f"{RUN_DIR}/manifest.json",              f"{RUN_NAME}_manifest.json")]:
        if os.path.exists(p):
            DRIVE.put(p, n)
    print("   results copied to Drive")

print(f"\n✅ {RUN_NAME} complete. Snapshot + results are in the Kaggle dataset "
      f"'{CKPT_DATASET_SLUG}' and your Drive folder.")
